<a href="https://colab.research.google.com/github/lydiacyhung/114-2-ProgramingLanguage/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

### 分析消費習慣

首先，我們將對 DataFrame 中的「金額」欄位進行處理，確保它是數值類型，然後根據「分類」來計算總花費。

In [ ]:
# 確保 '金額' 欄位是數值類型，如果轉換失敗則設為 NaN
df['金額'] = pd.to_numeric(df['金額'], errors='coerce')

# 移除金額為 NaN 的行，或可以選擇其他處理方式
df_cleaned_for_analysis = df.dropna(subset=['金額'])

# 計算各分類的總金額
category_spending = df_cleaned_for_analysis.groupby('分類')['金額'].sum().sort_values(ascending=False)
display(category_spending)

### 使用 Gemini AI 總結習慣與建議

現在我們將設置 Gemini API，並利用剛剛分析的消費數據，讓 AI 提供總結和建議。

In [ ]:
# 檢查是否已安裝 google-generativeai 庫，如果沒有則安裝
try:
    import google.generativeai as genai
except ImportError:
    !pip install -q -U google-generativeai
    import google.generativeai as genai

from google.colab import userdata

# 設定您的 Google API Key
# 請確保您的 GOOGLE_API_KEY 已儲存到 Colab Secrets 中 (左側面板的鑰匙圖標)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# 選擇一個模型
gemini_model = genai.GenerativeModel('gemini-pro')

# 準備給 AI 的提示詞
prompt = f"""
我提供你我的消費紀錄分類及各類別的總金額，請你根據這些數據，總結我的消費習慣，並提出一些具體的建議來改善或維持這些習慣。

我的消費數據如下：
{category_spending.to_string()}

請以條列式方式呈現消費習慣總結和建議。
"""

# 呼叫 Gemini 模型
response = gemini_model.generate_content(prompt)
print(response.text)

In [28]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [62]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1fc5laOITajG78TNIXouEP2ernmL9MKvi5VJb2jVWQyw/edit?usp=sharing')

In [63]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2025-09-11,9:30,,早餐三明治,50,,,,
1,2025-09-16,,交通,捷運,20,Pecu,,,
2,2025-09-18,,外食,早餐,80,Pecu,,,
3,2025-09-26,,交通,車費儲值,100,H,,,
4,2026-03-05,9:15,水果,apple,20,0,0,0,0


In [64]:
import datetime

In [66]:
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')

time_str = input("請輸入時間 (格式：HH:MM): ")
# 檢查時間格式是否正確
datetime.datetime.strptime(time_str, '%H:%M')
category=input("請輸入分類: ")
item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))
payer=input("請輸入付款人: ")

請輸入日期 (格式：YYYY-MM-DD): 2026-03-05
請輸入時間 (格式：HH:MM): 10:40
請輸入分類: 早餐
請輸入品項: 三明治
請輸入金額: 60
請輸入付款人: Lydia


In [67]:
date_str

'2026-03-05'

In [68]:
time_str

'10:40'

In [69]:
item

'三明治'

In [70]:
amount

60.0

In [71]:
new_data = pd.DataFrame([{
    '日期': date_str,
    '時間': time_str,
    '品項': item,
    '金額': amount,
    '分類':category,
    '付款人':payer,
    '備註':'NO',
    '地點':'NO',
    '支付方式':'NO'
    }])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [72]:
df

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2025-09-11,9:30,,早餐三明治,50,,,,
1,2025-09-16,,交通,捷運,20,Pecu,,,
2,2025-09-18,,外食,早餐,80,Pecu,,,
3,2025-09-26,,交通,車費儲值,100,H,,,
4,2026-03-05,9:15,水果,apple,20,0,0,0,0
5,2026-03-05,10:40,早餐,三明治,60.0,Lydia,NO,NO,NO


In [73]:
# 將 DataFrame 中的 NaN 值替換為空字串，以符合 JSON 格式要求
df_cleaned = df.fillna('')

# 將 cleaned DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df_cleaned.values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 第二步：將資料寫入工作表
# 因為我們是新增所有 df 的內容，所以將整份 df_cleaned 寫入，而不是單獨的 new_data
# 注意：worksheet.append_rows() 每次調用都會追加，如果之前有寫過，可能會重複添加
# 為了避免重複，通常會先清空或指定範圍寫入
# 這裡假設每次執行是追加新的 data_to_write 列表，而不是將整個 df 重複寫入
# 如果是第一次寫入整個 df, 應該這樣做：worksheet.update([df.columns.tolist()] + data_to_write)

# 這裡我們只追加 DataFrame 中新增的最新一筆資料，而不是整個 DataFrame
# 因為 df 已經包含了新舊資料，所以我們只取 df 的最後一行來追加
last_row_data = df_cleaned.iloc[[-1]].values.tolist()
worksheet.append_rows(values=last_row_data, value_input_option='USER_ENTERED')
print("最新一筆資料已成功寫入 Google 工作表！")


最新一筆資料已成功寫入 Google 工作表！
